In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, accuracy_score, roc_curve, auc, ConfusionMatrixDisplay

In [ ]:
df_original = pd.read_stata('Mexico-2023-full-data.dta')

# Seleccionar y renombrar columnas clave
cols = {'d2': 'sales', 'l1': 'employees', 'b5': 'start_year', 
        'l10': 'training', 'k82': 'financing', 'd3c': 'export_share'}

df_clean = df_original[list(cols.keys())].copy()
df_clean.rename(columns=cols, inplace=True)

# Convertir columnas a numéricas (lo que no sea número se vuelve NaN)
for col in ['sales', 'employees', 'start_year', 'export_share']:
    df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')

In [ ]:
# --- DEFINIR VARIABLE BINARIA DE ÉXITO (Punto clave de la tarea) ---
# Éxito = 1 si las ventas son superiores al promedio, 0 si no.
promedio_ventas = df_clean['sales'].mean()
df_clean['successful'] = (df_clean['sales'] > promedio_ventas).astype(int)

# --- CONVERTIR CATEGORÍAS A BINARIO (1 y 0) ---
df_clean['training'] = df_clean['training'].map({'Yes': 1, 'No': 0})
df_clean['financing'] = df_clean['financing'].map({'Yes': 1, 'No': 0})
df_clean['exporter'] = (df_clean['export_share'] > 0).astype(int)

# Eliminar nulos para que el modelo no tenga errores
df_log_final = df_clean.dropna(subset=['sales', 'employees', 'training', 'financing', 'successful']).copy()

print(f"Dataset limpio. Total de empresas analizadas: {len(df_log_final)}")
display(df_log_final.head())

In [ ]:
# Definir variables Independientes (X) y Dependiente (y)
# Estas son las que causaban el error 'y_logic' en tu archivo
X_logic = df_log_final[['employees', 'training', 'financing', 'exporter']]
y_logic = df_log_final['successful']

# Crear el modelo y entrenarlo
modelo_log = LogisticRegression()
modelo_log.fit(X_logic, y_logic)

# Generar predicciones
y_pred_log = modelo_log.predict(X_logic)
y_prob_log = modelo_log.predict_proba(X_logic)[:, 1] # Probabilidades para la curva ROC

In [ ]:
print(f"\nPrecisión del modelo (Accuracy): {accuracy_score(y_logic, y_pred_log):.4f}")

# A. Matriz de Confusión
plt.figure(figsize=(6,4))
cm = confusion_matrix(y_logic, y_pred_log)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No Exitosa', 'Exitosa'])
disp.plot(cmap='Blues', values_format='d')
plt.title('Matriz de Confusión')
plt.show()

In [ ]:
fpr, tpr, _ = roc_curve(y_logic, y_prob_log)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6,4))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'Curva ROC (área = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlabel('Tasa de Falsos Positivos')
plt.ylabel('Tasa de Verdaderos Positivos')
plt.title('Evaluación con Curva ROC')
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.show()